# **Monitoring Amazonas Deforestation with UP42 and Sentinel-2**

---
## **Scenario**

**Green-Plant Management GmbH** is monitoring potential ***deforestation in the Amazonas region*** for a government project. This notebook demonstrates a prototype support workflow for discovering, ordering, inspecting, and analyzing [**Sentinel-2**](https://docs.up42.com/data/sentinel-2) imagery programmatically with the [**UP42 API**](https://docs.up42.com/developers).

---
## **Objective**

The notebook implements a compact, repeatable pipeline that:

- authenticates with UP42
- loads a polygon [Area of Interest (AOI) in the Amazonas](https://gist.github.com/diegoalarc/5999f4641804e3a38066c539c77e7066)
- searches the [UP42 catalog for Sentinel-2](https://docs.up42.com/data/sentinel-2) scenes
- filters results by time range and cloud coverage
- selects two representative timestamps
- places orders and inspects the delivered results through [STAC](https://docs.up42.com/developers/api-stac)
- downloads the relevant raster assets
- computes [NDVI](https://up42.com/blog/5-things-to-know-about-ndvi) and an NDVI difference map
- visualizes potential vegetation change

---
## **Why this workflow?**

The goal is to reflect a realistic _technical support_ workflow that a customer could adapt for operational monitoring:

1. discover suitable imagery
2. validate scene metadata
3. order the selected assets
4. inspect the delivery in STAC/storage
5. retrieve analysis-ready raster files
6. compute a vegetation index consistently across dates
7. visualize change in a reproducible way

In [ ]:
# Imports

import json
import os
import uuid
import zipfile
from pathlib import Path

import geojson
import geopandas as gpd
import matplotlib.pyplot as plt
from matplotlib.offsetbox import OffsetImage, AnnotationBbox
import numpy as np
import pandas as pd
import rasterio
from dotenv import load_dotenv
import up42

---
## **Configuration**

The main search, ordering, and tracking parameters are defined in one place so the workflow is easy to review and modify.

---

This includes:

- the catalog search date range
- the cloud coverage threshold
- the target collection and data product
- naming and tagging settings for orders
- order tracking settings

In [ ]:
# ----------------------------
# Configuration
# ----------------------------

# The time window is chosen to provide a small but meaningful Sentinel-2 time series for the selected AOI.

# The cloud threshold is set conservatively (10%) to improve the chance of retrieving scenes suitable for vegetation-index analysis.

START_DATE = "2024-06-01"
END_DATE = "2024-12-31"
MAX_CLOUD_COVER = 10

COLLECTION_NAME = "sentinel-2"
DATA_PRODUCT_NAME = "sentinel-2-level-2a"

TITLE_PREFIX = "amazonas-deforestation"
TAGS = ["up42-challenge", "sentinel-2", "amazonas"]

REPORT_TIME_SECONDS = 60

---
## **Project paths**

This section defines the main input, download, and output folders used in the notebook.

Using explicit project paths improves reproducibility and makes it easier to re-run the workflow from a _clean environment_ or adapt it to _another AOI_.

In [ ]:
# ----------------------------
# Project paths
# ----------------------------

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebook" else Path.cwd()

DATA_DIR = PROJECT_ROOT / "data"
AOI_DIR = DATA_DIR / "aoi"
DOWNLOAD_DIR = DATA_DIR / "downloads"
FIG_DIR = PROJECT_ROOT / "outputs" / "figures"

DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("AOI_DIR:", AOI_DIR)
print("DOWNLOAD_DIR:", DOWNLOAD_DIR)
print("FIG_DIR:", FIG_DIR)

---
## **Authentication**

The notebook authenticates with the ***UP42 Python SDK*** using credentials stored in a local **`.env`** file.

This keeps secrets out of the notebook & reflects a security practice for reproducible API workflows.

In [ ]:
# ----------------------------
# Load credentials and authenticate
# ----------------------------

load_dotenv(PROJECT_ROOT / ".env")

UP42_USERNAME = os.getenv("UP42_USERNAME")
UP42_PASSWORD = os.getenv("UP42_PASSWORD")

if not UP42_USERNAME or not UP42_PASSWORD:
    raise ValueError(
        "Missing UP42 credentials. Add UP42_USERNAME and UP42_PASSWORD to the local .env file."
    )

up42.authenticate(
    username=UP42_USERNAME,
    password=UP42_PASSWORD,
)

print("Authenticated successfully.")

---
## Why **Sentinel-2 Level-2A**?

_Sentinel-2 Level-2A_ is a good fit for this prototype because it provides free, multispectral, analysis-ready surface reflectance imagery at medium spatial resolution.

It is well suited for vegetation monitoring because:

- it includes the red and near-infrared bands required for [NDVI](https://up42.com/blog/5-things-to-know-about-ndvi)
- it supports repeated observations over time
- Level-2A surface reflectance is more appropriate than raw top-of-atmosphere imagery for comparing vegetation conditions across dates

---
## **Area of Interest (AOI)**

The AOI is stored as a local [GeoJSON polygon](https://gist.github.com/diegoalarc/5999f4641804e3a38066c539c77e7066) in the Amazonas region.

Keeping the AOI separate from the notebook makes the workflow easier to reuse: the same code can be rerun for a different monitoring area simply by replacing the polygon file.

In [ ]:
# ----------------------------
# Load AOI
# ----------------------------

aoi_path = AOI_DIR / "amazonas_aoi.geojson"

if not aoi_path.exists():
    raise FileNotFoundError(f"AOI file not found: {aoi_path}")

aoi_gdf = gpd.read_file(aoi_path)

if aoi_gdf.empty:
    raise ValueError("AOI file was loaded, but it contains no features.")

if len(aoi_gdf) != 1:
    raise ValueError("Expected exactly one AOI feature in the GeoJSON file.")

if aoi_gdf.crs is None:
    raise ValueError("AOI GeoJSON has no CRS defined.")

display(aoi_gdf)

# Convert the AOI into a GeoJSON geometry dictionary for UP42 catalog search and orders
aoi_geometry = json.loads(aoi_gdf.to_json())["features"][0]["geometry"]

print(json.dumps(aoi_geometry, indent=2))

---
## **Product discovery**

Before placing any order, the notebook inspects the UP42 product glossary to confirm that the target archive collection and data product are available.

This keeps the workflow explicit and support-oriented:

- it avoids hardcoding hidden assumptions
- it shows how a user can verify the available collection structure
- it makes product and provider selection easier to troubleshoot

In [ ]:
# ----------------------------
# Retrieve archive collections
# ----------------------------

archive_collections = up42.ProductGlossary.get_collections(
    collection_type=up42.CollectionType.ARCHIVE,
    sort_by=up42.CollectionSorting.name.asc,
)

sentinel_collection = next(
    (c for c in archive_collections if c.name == COLLECTION_NAME),
    None,
)

if sentinel_collection is None:
    raise ValueError(f"Collection not found: {COLLECTION_NAME}")

print("Collection:", sentinel_collection.name)

print("\nAvailable data products:")
for dp in sentinel_collection.data_products:
    print("-", dp.name)

print("\nAvailable providers:")
for p in sentinel_collection.providers:
    print("-", p.name, "| host:", p.is_host)

# Select the data product used later for ordering
data_product = next(
    (dp for dp in sentinel_collection.data_products if dp.name == DATA_PRODUCT_NAME),
    None,
)

if data_product is None:
    raise ValueError(f"Data product not found: {DATA_PRODUCT_NAME}")

print("\nSelected data product:", data_product.name)
print("Data product ID:", data_product.id)

# Select the host provider used for catalog search
host = next((p for p in sentinel_collection.providers if p.is_host), None)

if host is None:
    raise ValueError("No host provider found for the selected collection.")

print("Selected host provider:", host.name)

---
## Catalog search

The catalog search is constrained by:

- the AOI polygon
- a defined date range
- a maximum scene-level cloud coverage threshold

This is the first filtering stage in the workflow. It reduces the candidate scene set to images that are more likely to be useful for vegetation analysis, while still preserving a small time series for later selection.

In [ ]:
# ----------------------------
# Search the catalog
# ----------------------------

search_results = list(
    host.search(
        collections=[COLLECTION_NAME],
        intersects=aoi_geometry,
        start_date=START_DATE,
        end_date=END_DATE,
        query={"cloudCoverage": {"LT": MAX_CLOUD_COVER}},
    )
)

scenes = search_results

print(f"Number of scenes found: {len(scenes)}")

if not scenes:
    raise ValueError(
        f"No scenes found for AOI, date range {START_DATE} to {END_DATE}, "
        f"and cloud cover < {MAX_CLOUD_COVER}%. Try widening the search."
    )

---
## **Search result summary**

The search results are converted into a compact metadata table so that scene selection is transparent and reviewable.

This is useful in a support workflow because it makes the selection logic visible rather than hiding it in later code cells.

In [ ]:
# ----------------------------
# Build a table of scene metadata
# ----------------------------

records = []
for item in scenes:
    records.append({
        "scene_id": item.id,
        "datetime": item.datetime,
        "cloud_coverage": item.cloud_coverage,
        "producer": item.producer,
        "constellation": item.constellation,
        "collection": item.collection,
        "resolution_m": item.resolution,
    })

scenes_df = (
    pd.DataFrame(records)
    .sort_values("datetime")
    .reset_index(drop=True)
)

display(scenes_df)

---
## **Time-series context**

The filtered search results form a small Sentinel-2 time series over the AOI.

For the raster-processing part of the prototype, the notebook uses two representative timestamps. This keeps the workflow compact while still demonstrating the full sequence of steps:

1. search the catalog
2. review scene metadata
3. select scenes
4. order the data
5. inspect the delivery through STAC
6. download raster assets
7. compute NDVI and change

This means the notebook demonstrates both:

- a **time-series discovery workflow** at catalog level
- a **two-date analysis workflow** at raster level

In [ ]:
# ----------------------------
# Time-series metadata view
# ----------------------------

scenes_df["datetime"] = pd.to_datetime(scenes_df["datetime"], errors="coerce")
scenes_df = scenes_df.sort_values("datetime").reset_index(drop=True)
scenes_df["date"] = scenes_df["datetime"].dt.date

scenes_ts_df = scenes_df[
    ["scene_id", "date", "datetime", "cloud_coverage", "producer"]
].copy()

display(scenes_ts_df)

---
## **Scene selection**

To keep the prototype simple and interpretable, two scenes are selected from the filtered time series for downstream raster analysis.

In this notebook, the earliest and latest suitable scenes are chosen to maximize temporal separation within the filtered result set. This is a reasonable prototype strategy for demonstrating change detection, although a production workflow could apply more advanced ranking criteria such as AOI-specific cloud conditions, seasonal comparability, or scene quality metrics.

In [ ]:
# ----------------------------
# Select two scenes
# ----------------------------

scene_1_idx = 0                    # earliest scene
scene_2_idx = len(scenes_df) - 1   # latest scene

scene_1_id = scenes_df.loc[scene_1_idx, "scene_id"]
scene_2_id = scenes_df.loc[scene_2_idx, "scene_id"]

scene_1 = next(s for s in scenes if s.id == scene_1_id)
scene_2 = next(s for s in scenes if s.id == scene_2_id)

print("Scene 1")
print("ID:", scene_1.id)
print("Datetime:", scene_1.datetime)
print("Cloud coverage:", scene_1.cloud_coverage)
print("Producer:", scene_1.producer)

print("\nScene 2")
print("ID:", scene_2.id)
print("Datetime:", scene_2.datetime)
print("Cloud coverage:", scene_2.cloud_coverage)
print("Producer:", scene_2.producer)

---
## **Order preparation**

After identifying two candidate scenes, the notebook converts them into _UP42_ order templates.

Each order template includes:

- the selected Sentinel-2 data product
- the AOI geometry
- the selected scene identifier
- tags for traceability
- a unique display name for later lookup in storage

Using unique titles is helpful in a support workflow because it creates a clear link between the order submitted to the platform and the STAC item retrieved later from storage.

In [ ]:
# ----------------------------
# Helper function: create an order template
# ----------------------------

def make_order_template(scene, geometry, data_product, title_prefix=TITLE_PREFIX, tags=TAGS):
    if scene is None:
        raise ValueError("Scene must not be None.")
    if geometry is None:
        raise ValueError("AOI geometry must not be None.")
    if data_product is None:
        raise ValueError("Data product must not be None.")

    # UP42 expects the AOI as a FeatureCollection
    aoi = geojson.FeatureCollection(
        features=[geojson.Feature(geometry=geometry)]
    )

    # Unique display names simplify lookup of the fulfilled item in STAC storage
    unique_title = f"{title_prefix}-{scene.id}-{uuid.uuid4().hex[:8]}"

    order_template = up42.BatchOrderTemplate(
        data_product_id=data_product.id,
        display_name=unique_title,
        features=aoi,
        params={"id": scene.id},
        tags=tags,
    )

    return order_template, unique_title

In [ ]:
# ----------------------------
# Create order templates for the two selected scenes
# ----------------------------

order_template_1, order_title_1 = make_order_template(
    scene_1, aoi_geometry, data_product
)
order_template_2, order_title_2 = make_order_template(
    scene_2, aoi_geometry, data_product
)

print("Order title 1:", order_title_1)
print("Order title 2:", order_title_2)

---
## **Order estimate check**

Before placing the orders, the notebook inspects the estimated order details.

This is a useful validation step because it helps confirm that the request is correctly configured before submitting it to the platform.

In [ ]:
# ----------------------------
# Inspect order estimates
# ----------------------------

print("Scene 1 estimate:")
print(order_template_1.estimate)

print("\nScene 2 estimate:")
print(order_template_2.estimate)

## **Note on order placement prerequisites**

Catalog search may succeed even when order placement is blocked by account-level prerequisites.

During testing, order placement initially failed because the required EULA had not yet been accepted in the UP42 Console. After accepting the pending EULA in the workspace settings, the order workflow could proceed normally.

This is an important support consideration: failures at the ordering stage are not always caused by notebook code. They may also be related to workspace permissions, legal acceptance, or entitlement settings.

In [ ]:
# ----------------------------
# Helper function: place an order
# ----------------------------

def place_order(order_template, label):
    try:
        order_refs = order_template.place()

        if not order_refs:
            raise RuntimeError(f"{label} returned no order references.")

        order = order_refs[0].order

        print(f"{label} order placed successfully.")
        print("Order ID:", order.id)
        print("Status:", order.status)

        return order

    except Exception as e:
        raise RuntimeError(f"{label} order placement failed: {e}") from e

In [ ]:
# ----------------------------
# Place both orders
# ----------------------------

order_1 = place_order(order_template_1, "Scene 1")
order_2 = place_order(order_template_2, "Scene 2")

In [ ]:
# ----------------------------
# Helper function: track order until terminal state
# ----------------------------

def track_order(order, label, report_time=REPORT_TIME_SECONDS):
    print(f"Tracking {label}...")
    order.track(report_time=report_time)

    print(f"{label} final status: {order.status}")
    print(f"{label} fulfilled: {order.is_fulfilled}")

    if not order.is_fulfilled:
        raise RuntimeError(f"{label} did not complete successfully.")

In [ ]:
# ----------------------------
# Track both orders
# ----------------------------

track_order(order_1, "Scene 1 order")
track_order(order_2, "Scene 2 order")

---
## **STAC** inspection of fulfilled outputs

Once an order is fulfilled, the resulting delivery becomes accessible in UP42 storage through STAC-compatible objects.

This section demonstrates how to:

- connect to the UP42 STAC client
- search storage for the fulfilled outputs
- retrieve the matching STAC items
- inspect item metadata and available assets

This step is important because it connects the ordering workflow to downstream asset retrieval and raster analysis.

In [ ]:
# ----------------------------
# Create STAC client
# ----------------------------

stac_client = up42.stac_client()

# ----------------------------
# Helper function: search STAC items by unique order title
# ----------------------------

def search_stac_items_by_title(unique_title):
    filter_ = {
        "op": "a_contains",
        "args": [
            {"property": "up42-user:title"},
            unique_title,
        ],
    }

    search = stac_client.search(filter=filter_)
    return list(search.items())


# ----------------------------
# Helper function: choose the best matching STAC item
# ----------------------------

def select_best_stac_item(items, unique_title):
    if not items:
        raise ValueError(
            f"No STAC item found for title '{unique_title}'. "
            "Order may still be processing or indexing is not complete."
        )

    # Prefer exact match
    exact_matches = [
        item for item in items
        if item.properties.get("up42-user:title") == unique_title
    ]
    candidate_items = exact_matches if exact_matches else items

    # Sort by datetime safely
    candidate_items = sorted(
        candidate_items,
        key=lambda x: x.datetime if x.datetime else 0,
        reverse=True,
    )

    return candidate_items[0]


# ----------------------------
# Helper function: summarize a STAC item
# ----------------------------

def summarize_stac_item(item):
    return {
        "id": item.id,
        "datetime": item.datetime,
        "collection": getattr(item, "collection_id", None),
        "platform": item.properties.get("platform"),
        "constellation": item.properties.get("constellation"),
        "gsd": item.properties.get("gsd"),
        "asset_keys": sorted(item.assets.keys()),
        "n_assets": len(item.assets),
        "has_geometry": item.geometry is not None,
        "bbox": getattr(item, "bbox", None),
        "title": item.properties.get("up42-user:title"),
    }


# ----------------------------
# Helper function: validate required assets
# ----------------------------

# Mapping for Sentinel-2 bands
BAND_ALIASES = {
    "red": ["red", "b04", "b04.tiff"],
    "nir": ["nir", "b08", "b08.tiff"],
}

def validate_required_assets(item, required_assets=None):
    required_assets = required_assets or []
    available_assets = set(item.assets.keys())

    missing_assets = []

    for req in required_assets:
        aliases = BAND_ALIASES.get(req, [req])
        found = any(alias in available_assets for alias in aliases)

        if not found:
            missing_assets.append(req)

    if missing_assets:
        raise ValueError(
            f"Item '{item.id}' is missing required assets: {missing_assets}. "
            f"Available assets: {sorted(available_assets)}"
        )

    return True


# ----------------------------
# Helper function: get asset key
# ----------------------------

def get_asset_key(item, band_name):
    available_assets = set(item.assets.keys())
    aliases = BAND_ALIASES.get(band_name, [band_name])

    for alias in aliases:
        if alias in available_assets:
            return alias

    raise ValueError(f"No asset found for band '{band_name}' in item {item.id}")


# ----------------------------
# Helper function: print a readable summary
# ----------------------------

def print_stac_summary(summary, label):
    print(label)
    print("-" * len(label))
    for key, value in summary.items():
        print(f"{key}: {value}")


In [ ]:
# ----------------------------
# Find the STAC items corresponding to the placed orders
# ----------------------------
items_1 = search_stac_items_by_title(order_title_1)
items_2 = search_stac_items_by_title(order_title_2)

stac_item_1 = select_best_stac_item(items_1, order_title_1)
stac_item_2 = select_best_stac_item(items_2, order_title_2)

# Validate required assets (NOW WORKS)
validate_required_assets(stac_item_1, required_assets=["red", "nir"])
validate_required_assets(stac_item_2, required_assets=["red", "nir"])

# Get correct asset keys (CRUCIAL FIX)
red_key_1 = get_asset_key(stac_item_1, "red")
nir_key_1 = get_asset_key(stac_item_1, "nir")

red_key_2 = get_asset_key(stac_item_2, "red")
nir_key_2 = get_asset_key(stac_item_2, "nir")

print(f"Scene 1 → red: {red_key_1}, nir: {nir_key_1}")
print(f"Scene 2 → red: {red_key_2}, nir: {nir_key_2}")

# Summaries
summary_1 = summarize_stac_item(stac_item_1)
summary_2 = summarize_stac_item(stac_item_2)

print_stac_summary(summary_1, "Scene 1 STAC summary")
print()
print_stac_summary(summary_2, "Scene 2 STAC summary")

## **STAC** validation outcome

The fulfilled outputs were successfully located in storage and inspected through the UP42 STAC client.

At this point, the workflow has confirmed that:

- the expected fulfilled items can be retrieved from storage
- the item metadata is accessible
- the required raster assets for NDVI analysis are available

This validation step reduces the risk of failing later during raster download or band-level processing.

In [ ]:
# ----------------------------
# Inspect selected STAC properties
# ----------------------------

def print_selected_properties(item, property_keys, label="STAC item"):
    print(f"{label} selected properties:")
    for key in property_keys:
        print(f"{key}: {item.properties.get(key)}")

print()
print_selected_properties(
    stac_item_1,
    ["datetime", "platform", "constellation", "gsd", "up42-user:title"],
    label="Scene 1"
)

print("\nFirst 20 property keys:")
print(list(stac_item_1.properties.keys())[:20])

---
## **Download the original delivery**

For this prototype, the original fulfilled delivery is downloaded and extracted locally before raster processing.

This approach keeps the workflow easy to inspect end to end:

- the fulfilled UP42 delivery is preserved as downloaded
- the local extraction step is explicit and reproducible
- the relationship between the platform delivery and the analysis-ready raster files remains transparent

In a production workflow, it may also be possible to work directly with selected STAC assets, but downloading the original delivery is a simple starting point for a support-oriented prototype.

In [ ]:
# ----------------------------
# Helper function: download the original delivery
# ----------------------------

def download_original_delivery(stac_item, out_dir):
    """
    Download the original fulfilled delivery associated with a STAC item.
    The function first checks collection-level assets, then item-level assets.
    """
    collection = stac_item.get_collection()

    # Try collection-level assets first
    original_delivery = next(
        (
            asset
            for asset in collection.assets.values()
            if asset.roles and "original" in asset.roles
        ),
        None,
    )

    # Fallback: try item-level assets
    if original_delivery is None:
        original_delivery = next(
            (
                asset
                for asset in stac_item.assets.values()
                if asset.roles and "original" in asset.roles
            ),
            None,
        )

    if original_delivery is None:
        raise ValueError(
            f"No original delivery asset found for STAC item '{stac_item.id}'."
        )

    downloaded_file_path = original_delivery.file.download(output_directory=out_dir)
    return Path(downloaded_file_path)

In [ ]:
# ----------------------------
# Download the original deliveries
# ----------------------------

zip_path_1 = download_original_delivery(stac_item_1, DOWNLOAD_DIR)
zip_path_2 = download_original_delivery(stac_item_2, DOWNLOAD_DIR)

print("Scene 1 ZIP:", zip_path_1)
print("Scene 2 ZIP:", zip_path_2)

In [ ]:
# ----------------------------
# Helper function: unzip a delivery
# ----------------------------

def unzip_file(zip_path, output_folder):
    """
    Extract a ZIP file safely into a folder.
    Ensures files do not escape the target directory.
    """
    output_folder.mkdir(parents=True, exist_ok=True)
    output_folder = output_folder.resolve()

    with zipfile.ZipFile(zip_path, "r") as z:
        for member in z.infolist():
            member_path = (output_folder / member.filename).resolve()

            if not str(member_path).startswith(str(output_folder)):
                raise ValueError(f"Unsafe ZIP entry detected: {member.filename}")

            if member.is_dir():
                member_path.mkdir(parents=True, exist_ok=True)
            else:
                member_path.parent.mkdir(parents=True, exist_ok=True)
                with z.open(member) as source, open(member_path, "wb") as target:
                    target.write(source.read())

    return output_folder

In [ ]:
# ----------------------------
# Extract the downloaded deliveries
# ----------------------------

scene1_dir = unzip_file(zip_path_1, DOWNLOAD_DIR / f"scene1_{scene_1.id}")
scene2_dir = unzip_file(zip_path_2, DOWNLOAD_DIR / f"scene2_{scene_2.id}")

print("Scene 1 extracted to:", scene1_dir)
print("Scene 2 extracted to:", scene2_dir)

In [ ]:
# ----------------------------
# Helper function: inspect extracted files
# ----------------------------

def inspect_extracted_files(folder, label, n_preview=10):
    all_files = [p for p in folder.rglob("*") if p.is_file()]
    print(f"Extracted {len(all_files)} files for {label}.")
    print("Example files:")
    for p in all_files[:n_preview]:
        print("-", p.name)
    return all_files

all_scene1_files = inspect_extracted_files(scene1_dir, "Scene 1")
print()
all_scene2_files = inspect_extracted_files(scene2_dir, "Scene 2")

In [ ]:
# ----------------------------
# Helper function: locate a band-specific GeoTIFF
# ----------------------------

def find_band_file(folder, band_code):
    matches = sorted(
        [
            p for p in folder.rglob("*.tif")
            if band_code in p.name
            and "thumbnail" not in p.name.lower()
            and "preview" not in p.name.lower()
            and "visual" not in p.name.lower()
        ]
    )

    if not matches:
        raise FileNotFoundError(f"No TIFF found for {band_code} in {folder}")

    if len(matches) > 1:
        print(f"Multiple matches found for {band_code}:")
        for m in matches:
            print(" -", m)
        print("Using first match in sorted order.")

    return matches[0]

In [ ]:
# ----------------------------
# Find red and NIR files for both scenes
# ----------------------------

red1_path = find_band_file(scene1_dir, "B04")
nir1_path = find_band_file(scene1_dir, "B08")

red2_path = find_band_file(scene2_dir, "B04")
nir2_path = find_band_file(scene2_dir, "B08")

print("Scene 1 red:", red1_path)
print("Scene 1 nir:", nir1_path)
print("Scene 2 red:", red2_path)
print("Scene 2 nir:", nir2_path)

---
## Why **NDVI**?

NDVI is a simple and widely used vegetation index:

$NDVI = \frac{NIR - Red}{NIR + Red}$

For Sentinel-2:

- **Band 8 (B08)** provides near-infrared reflectance
- **Band 4 (B04)** provides red reflectance

Healthy vegetation typically reflects strongly in the near-infrared and absorbs more in the red range, so NDVI is a useful first indicator of vegetation condition and potential change over time.

In [ ]:
# ----------------------------
# Helper functions: read raster, apply scaling, compute NDVI
# ----------------------------

def read_band(path):
    """
    Read a single-band raster and return the array, profile, nodata value,
    and any scale/offset metadata.
    """
    with rasterio.open(path) as src:
        arr = src.read(1).astype("float32")
        profile = src.profile
        nodata = src.nodata
        scales = getattr(src, "scales", None)
        offsets = getattr(src, "offsets", None)
    return arr, profile, nodata, scales, offsets


def apply_scale_and_offset(arr, scale=None, offset=None):
    """
    Apply raster scale and offset if provided.
    """
    if scale is not None:
        arr = arr * scale
    if offset is not None:
        arr = arr + offset
    return arr


def compute_ndvi(red, nir, red_nodata=None, nir_nodata=None):
    """
    Compute NDVI from red and NIR arrays while masking invalid pixels.
    """
    invalid_mask = np.zeros(red.shape, dtype=bool)

    if red_nodata is not None:
        invalid_mask |= (red == red_nodata)
    if nir_nodata is not None:
        invalid_mask |= (nir == nir_nodata)

    invalid_mask |= (red <= 0) | (nir <= 0)

    denominator = nir + red
    invalid_mask |= (denominator == 0)

    ndvi = np.full(red.shape, np.nan, dtype="float32")
    ndvi[~invalid_mask] = (
        (nir[~invalid_mask] - red[~invalid_mask]) /
        denominator[~invalid_mask]
    )

    return ndvi

In [ ]:
# ----------------------------
# Read and prepare bands
# ----------------------------

# Scene 1
red1, profile1, red1_nodata, red1_scales, red1_offsets = read_band(red1_path)
nir1, _, nir1_nodata, nir1_scales, nir1_offsets = read_band(nir1_path)

red1 = apply_scale_and_offset(
    red1,
    scale=red1_scales[0] if red1_scales else None,
    offset=red1_offsets[0] if red1_offsets else None,
)
nir1 = apply_scale_and_offset(
    nir1,
    scale=nir1_scales[0] if nir1_scales else None,
    offset=nir1_offsets[0] if nir1_offsets else None,
)

# Scene 2
red2, profile2, red2_nodata, red2_scales, red2_offsets = read_band(red2_path)
nir2, _, nir2_nodata, nir2_scales, nir2_offsets = read_band(nir2_path)

red2 = apply_scale_and_offset(
    red2,
    scale=red2_scales[0] if red2_scales else None,
    offset=red2_offsets[0] if red2_offsets else None,
)
nir2 = apply_scale_and_offset(
    nir2,
    scale=nir2_scales[0] if nir2_scales else None,
    offset=nir2_offsets[0] if nir2_offsets else None,
)

print("Scene 1 band shapes:", red1.shape, nir1.shape)
print("Scene 2 band shapes:", red2.shape, nir2.shape)

# ----------------------------
# Compute NDVI for both scenes
# ----------------------------

ndvi1 = compute_ndvi(red1, nir1, red1_nodata, nir1_nodata)
ndvi2 = compute_ndvi(red2, nir2, red2_nodata, nir2_nodata)

print("Scene 1 NDVI min/max:", np.nanmin(ndvi1), np.nanmax(ndvi1))
print("Scene 2 NDVI min/max:", np.nanmin(ndvi2), np.nanmax(ndvi2))

---
## **NDVI** difference map

The NDVI difference is computed as:

**NDVI difference = NDVI at Date 2 - NDVI at Date 1**

Negative values may indicate vegetation loss or disturbance, while positive values may indicate regrowth, seasonal variation, or other changes in surface conditions.

In [ ]:
# ----------------------------
# Validate spatial alignment before pixel-wise differencing
# ----------------------------

def validate_alignment(profile1, profile2, label1="Raster 1", label2="Raster 2"):
    if profile1["crs"] != profile2["crs"]:
        raise ValueError(
            f"{label1} and {label2} have different CRS: "
            f"{profile1['crs']} vs {profile2['crs']}"
        )

    if profile1["transform"] != profile2["transform"]:
        raise ValueError(
            f"{label1} and {label2} have different affine transforms."
        )

    if (profile1["height"], profile1["width"]) != (profile2["height"], profile2["width"]):
        raise ValueError(
            f"{label1} and {label2} have different shapes: "
            f"{(profile1['height'], profile1['width'])} vs {(profile2['height'], profile2['width'])}"
        )

validate_alignment(profile1, profile2, label1="Scene 1", label2="Scene 2")
print("Spatial alignment check passed.")

# Compute the difference map
ndvi_diff = ndvi2 - ndvi1

print("NDVI difference min/max:", np.nanmin(ndvi_diff), np.nanmax(ndvi_diff))

In [ ]:
# ----------------------------
# Helper function: plot and save a raster array
# ----------------------------

def plot_array(arr, title, output_path, cmap="RdYlGn", vmin=None, vmax=None,
               nodata_color="lightgray", colorbar_label=None):
    masked_arr = np.ma.masked_invalid(arr)

    cmap_obj = plt.get_cmap(cmap).copy()
    cmap_obj.set_bad(color=nodata_color)

    plt.figure(figsize=(8, 6))
    im = plt.imshow(masked_arr, cmap=cmap_obj, vmin=vmin, vmax=vmax)
    plt.colorbar(im, label=colorbar_label if colorbar_label else "Value")
    plt.title(title)
    plt.axis("off")
    plt.tight_layout()
    plt.savefig(output_path, dpi=200, bbox_inches="tight")
    plt.show()
    plt.close()

In [ ]:
# ----------------------------
# Visualize and save results
# ----------------------------

date_label_1 = pd.to_datetime(scene_1.datetime).strftime("%d %b %Y")
date_label_2 = pd.to_datetime(scene_2.datetime).strftime("%d %b %Y")

plot_array(
    ndvi1,
    f"NDVI - {date_label_1}",
    FIG_DIR / "ndvi_date1.png",
    cmap="RdYlGn",
    vmin=-1,
    vmax=1,
    colorbar_label="NDVI",
)

plot_array(
    ndvi2,
    f"NDVI - {date_label_2}",
    FIG_DIR / "ndvi_date2.png",
    cmap="RdYlGn",
    vmin=-1,
    vmax=1,
    colorbar_label="NDVI",
)

plot_array(
    ndvi_diff,
    f"NDVI Difference: {date_label_2} - {date_label_1}",
    FIG_DIR / "ndvi_difference.png",
    cmap="RdBu",
    vmin=-1,
    vmax=1,
    colorbar_label="NDVI difference",
)

In [ ]:
# ----------------------------
# Helper function: save a single-band array as GeoTIFF
# ----------------------------

def save_geotiff(arr, reference_profile, output_path, nodata_value=-9999.0):
    profile = reference_profile.copy()

    out_arr = np.where(np.isnan(arr), nodata_value, arr).astype("float32")

    profile.update(
        driver="GTiff",
        dtype="float32",
        count=1,
        nodata=nodata_value,
        compress="lzw"
    )

    with rasterio.open(output_path, "w", **profile) as dst:
        dst.write(out_arr, 1)

In [ ]:
NDVI1_TIF = FIG_DIR / "ndvi_date1.tif"
NDVI2_TIF = FIG_DIR / "ndvi_date2.tif"
NDVI_DIFF_TIF = FIG_DIR / "ndvi_difference.tif"

save_geotiff(ndvi1, profile1, NDVI1_TIF)
save_geotiff(ndvi2, profile2, NDVI2_TIF)
save_geotiff(ndvi_diff, profile1, NDVI_DIFF_TIF)

print("Saved:", NDVI1_TIF)
print("Saved:", NDVI2_TIF)
print("Saved:", NDVI_DIFF_TIF)

In [ ]:
# ----------------------------
# Create a 3-panel summary figure
# ----------------------------

def create_summary_panel(
    ndvi1,
    ndvi2,
    ndvi_diff,
    date_label_1,
    date_label_2,
    output_path,
    logo_path=None,
    nodata_color="lightgray",
):
    """
    Create a 3-panel figure:
    1. NDVI at Date 1
    2. NDVI at Date 2
    3. NDVI difference (Date 2 - Date 1)

    """

    # Use masked arrays so NaNs are rendered explicitly
    ndvi1_masked = np.ma.masked_invalid(ndvi1)
    ndvi2_masked = np.ma.masked_invalid(ndvi2)
    ndvi_diff_masked = np.ma.masked_invalid(ndvi_diff)

    # Create colormaps with explicit nodata color
    cmap_ndvi = plt.get_cmap("RdYlGn").copy()
    cmap_ndvi.set_bad(color=nodata_color)

    cmap_diff = plt.get_cmap("RdBu").copy()
    cmap_diff.set_bad(color=nodata_color)

    # Global figure style
    plt.rcParams.update({
        "font.size": 11,
        "axes.titlesize": 13,
        "axes.labelsize": 11,
        "figure.titlesize": 18,
        "savefig.dpi": 300,
    })

    fig, axes = plt.subplots(
        1, 3,
        figsize=(18, 6.5),
        constrained_layout=False
    )

    # Main title + subtitle
    fig.suptitle(
        "Amazonas Vegetation Change Analysis",
        fontweight="bold",
        y=0.98
    )
    fig.text(
        0.5, 0.93,
        "Sentinel-2 Level-2A | NDVI comparison across two timestamps | UP42 workflow prototype",
        ha="center",
        va="center",
        fontsize=11,
        alpha=0.85
    )

    # Plot 1
    im1 = axes[0].imshow(ndvi1_masked, cmap=cmap_ndvi, vmin=-1, vmax=1)
    axes[0].set_title(f"A. NDVI - {date_label_1}", fontweight="bold")
    axes[0].axis("off")

    # Plot 2
    im2 = axes[1].imshow(ndvi2_masked, cmap=cmap_ndvi, vmin=-1, vmax=1)
    axes[1].set_title(f"B. NDVI - {date_label_2}", fontweight="bold")
    axes[1].axis("off")

    # Plot 3
    im3 = axes[2].imshow(ndvi_diff_masked, cmap=cmap_diff, vmin=-1, vmax=1)
    axes[2].set_title(f"C. NDVI Difference: {date_label_2} - {date_label_1})", fontweight="bold")
    axes[2].axis("off")

    # Add individual colorbars
    cbar1 = fig.colorbar(im1, ax=axes[0], fraction=0.046, pad=0.04)
    cbar1.set_label("NDVI")

    cbar2 = fig.colorbar(im2, ax=axes[1], fraction=0.046, pad=0.04)
    cbar2.set_label("NDVI")

    cbar3 = fig.colorbar(im3, ax=axes[2], fraction=0.046, pad=0.04)
    cbar3.set_label("NDVI Difference")

    # Optional footer note
    fig.text(
        0.5, 0.03,
        "Gray areas indicate invalid or masked pixels.",
        ha="center",
        va="center",
        fontsize=10,
        alpha=0.75
    )

    # Optional logo stamp
    if logo_path is not None:
        logo_path = Path(logo_path)
        if logo_path.exists():
            try:
                logo_img = plt.imread(logo_path)
                imagebox = OffsetImage(logo_img, zoom=0.11)
                imagebox.set_alpha(0.5)
                
                ab = AnnotationBbox(
                    imagebox,
                    (0.975, 0.965),
                    xycoords="figure fraction",
                    frameon=False,
                    box_alignment=(1, 1)
                )
                fig.add_artist(ab)
            except Exception as e:
                print(f"Logo could not be added: {e}")

    # Spacing
    plt.subplots_adjust(
        left=0.04,
        right=0.98,
        top=0.86,
        bottom=0.10,
        wspace=0.18
    )

    # Save & show
    fig.savefig(output_path, bbox_inches="tight", facecolor="white")
    plt.show()
    plt.close(fig)


# Example output path
SUMMARY_FIG_PATH = FIG_DIR / "ndvi_summary_panel.png"

# Optional local logo path
UP42_LOGO_PATH = PROJECT_ROOT / "docs" / "screenshots" / "up42_logo.png"

create_summary_panel(
    ndvi1=ndvi1,
    ndvi2=ndvi2,
    ndvi_diff=ndvi_diff,
    date_label_1=date_label_1,
    date_label_2=date_label_2,
    output_path=SUMMARY_FIG_PATH,
    logo_path=UP42_LOGO_PATH,
)

print("Saved summary panel to:", SUMMARY_FIG_PATH)

## Interpretation

The two NDVI maps show vegetation conditions at the selected timestamps, and the difference map highlights areas where vegetation response changed between those dates.

In this prototype:

- negative NDVI differences may indicate possible vegetation loss or disturbance
- positive differences may reflect regrowth, seasonal variability, or other surface changes

The result should be treated as a compact proof of concept for vegetation-change screening rather than a full operational deforestation monitoring system.

In [ ]:
# ----------------------------
# Plot filtered scene availability over time
# ----------------------------

plot_df = scenes_ts_df.dropna(subset=["datetime", "cloud_coverage"]).copy()

plt.figure(figsize=(10, 4))
plt.plot(
    plot_df["datetime"],
    plot_df["cloud_coverage"],
    marker="o"
)

plt.title("Filtered Sentinel-2 scenes over time")
plt.xlabel("Acquisition date")
plt.ylabel("Cloud coverage (%)")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# ----------------------------
# Summarize scene counts by month
# ----------------------------

scenes_ts_df = scenes_ts_df.copy()
scenes_ts_df["year_month"] = scenes_ts_df["datetime"].dt.to_period("M").astype(str)

monthly_counts = (
    scenes_ts_df.groupby("year_month")
    .size()
    .reset_index(name="scene_count")
)

display(monthly_counts)

plt.figure(figsize=(10, 4))
plt.bar(monthly_counts["year_month"], monthly_counts["scene_count"])
plt.title("Number of filtered Sentinel-2 scenes per month")
plt.xlabel("Month")
plt.ylabel("Number of scenes")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## Why two timestamps were selected for raster analysis

The catalog search returns a filtered time series of candidate Sentinel-2 scenes over the AOI.

For the detailed raster-analysis section, two timestamps were selected to keep the prototype:

- computationally lightweight
- easy to audit
- easy to explain in a support context

This satisfies the requirement to compute an index at multiple timestamps while keeping the workflow compact and reproducible.

The notebook processes two representative timestamps in detail, but the same search, order, STAC, download, and NDVI logic could be extended to additional scenes in a production workflow.

In [ ]:
print("Workflow complete.")
print("Saved outputs:")

saved_files = sorted(
    [p for p in FIG_DIR.iterdir() if p.suffix.lower() in [".png", ".tif", ".tiff"]]
)

for p in saved_files:
    print("-", p.name)